<a href="https://colab.research.google.com/github/jugwr/crypto-pyspark-pipeline/blob/main/crypto_pipeline_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Setup Spark

In [ ]:
!pip install pyspark

Step 2: Extract from API

In [ ]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
import sqlite3

# # Initialize เปิดการทำงาน Spark
# print("Starting Spark Session")
# spark = SparkSession.builder \
#       .appName("CryptoDataPipeline") \
#       .getOrCreate()

# Initializw เปิดการทำงานพร้อมกับโหลด JDBC Driver
print("Starting Spark Session with JDBC Driver...")
# use config load Driver SQLite (Always did when Connect DB)
spark = SparkSession.builder \
      .appName("CryptoDataPipeline") \
      .config("spark.jars.packages", "org.xerial:sqlite-jdbc:3.34.0") \
      .getOrCreate()

# Extract Process
print("Extracting data from API...")
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 50,
    "page": 1,
    "sparkline": False
}

response = requests.get(url, params=params)
data = response.json() # ดึงข้อมูลออกมาเป็น JSON

# API Return Data "List of Dict" โครงสร้างซับซ้อน
# ใช้ pd แปลงก่อนนำเข้า spark ลดความยุ่งยากของ data schema
pdf = pd.DataFrame(data)
columns_to_keep = ["id", "symbol", "name", "current_price", "market_cap","total_volume", "last_updated"]
pdf = pdf[columns_to_keep]

# Transform Process (PySpark)
print("Transforming data with PySpark...")

sdf = spark.createDataFrame(pdf) # สร้าง Spark DataFrame

 # ลบค่าว่าง
sdf_cleaned = sdf.na.drop() \
    .withColumn("last_updated", to_timestamp(col("last_updated")))

# Result Transform n Schema
print("\n--- Cleaned Data (Sypark DataFrame) ---")
sdf_cleaned.show()

print("\n--- Data Schema ---")
sdf_cleaned.printSchema()

# # Load Process 1 save as .Parquet
# print("Loading data...")

# # In Case Big Data save as .Parquet (Less file and schema)
# output_path = "crypto_cleaned_data.parquet"
# sdf_cleaned.write.mode("overwrite").parquet(output_path)

# print(f"Pipeline success Data loaded into {output_path}")

# # Load Process (Database) 2 save into DB
# print("Loading data into database...")

# # Create URL database and Table name
# jdbc_url = "jdbc:sqlite:crypto_pyspark.db"
# table_name = "crypto_prices"

# # Put Data into Database by Spark
# sdf_cleaned.write \
#       .format("jdbc") \
#       .option("url", jdbc_url) \
#       .option("dbtable", table_name) \
#       .option("driver", "org.sqlite.JDBC") \
#       .mode("overwrite") \
#       .save()

# print("Pipeline success Data loaded into database 'crypto_pyspark.db' via JDBC.")

# Load Process 3 but use Spark to Pandas วิธีนำเข้าเหมือนใช้ Replit
df_final = sdf_cleaned.toPandas()

conn = sqlite3.connect('crypto_pyspark.db')

df_final.to_sql('crypto_prices', conn, if_exists='replace', index=False)
conn.close()

print("Pipeline success Data saved to 'crypto_pyspark.db' safely")

Opt: Run Result from DB

In [ ]:
df_from_db = spark.read \
      .format("jdbc") \
      .option("url", "jdbc:sqlite:crypto_pyspark.db") \
      .option("dbtable", "crypto_prices") \
      .option("driver", "org.sqlite.JDBC") \
      .load()

df_from_db.show()